# Mario RL — Ejecutar modelo entrenado
Carga un modelo `.zip` y lo corre con ventana visual.

In [1]:
import os, time
import numpy as np
import torch
from stable_baselines3 import PPO
from gym_utils import load_smb_env

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Dispositivo: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
# ── Configuración ─────────────────────────────────────────────
MODEL_PATH    = './data/models/59527849-f6cc-43b8-bc58-ac615c0965a9/final_model.zip'  # <-- cambia esto
WORLD         = 6
LEVEL         = 2
ACTION_SET    = 'COMPLEX'   # 'SIMPLE' | 'RIGHT' | 'COMPLEX'
CROP_DIM      = [0, 16, 0, 13]
N_STACK       = 6
N_SKIP        = 3
EPISODES      = 5
DETERMINISTIC = True
FRAME_DELAY   = 0.04   # segundos entre frames (~25 FPS)

In [3]:
# ── Cargar entorno y modelo ───────────────────────────────────
assert os.path.exists(MODEL_PATH), f'No se encontró: {MODEL_PATH}'

env_id   = f'SuperMarioBros-{WORLD}-{LEVEL}-v0'
vec_env  = load_smb_env(env_id, CROP_DIM, N_STACK, N_SKIP, ACTION_SET)
model    = PPO.load(MODEL_PATH, env=vec_env, device=device)

print(f'Modelo cargado OK')
print(f'Obs space: {vec_env.observation_space}')

/home/deynar/miniconda3/envs/mario_env/lib/python3.11/site-packages/gym/envs/registration.py:555: UserWarning: WARN: The environment SuperMarioBros-6-2-v0 is out of date. You should consider upgrading to version `v3`.
  logger.warn(
/home/deynar/miniconda3/envs/mario_env/lib/python3.11/site-packages/stable_baselines3/common/save_util.py:167: UserWarning: Could not deserialize object clip_range. Consider using `custom_objects` argument to replace this object.
Exception: code() argument 13 must be str, not int
  warnings.warn(
/home/deynar/miniconda3/envs/mario_env/lib/python3.11/site-packages/stable_baselines3/common/save_util.py:167: UserWarning: Could not deserialize object lr_schedule. Consider using `custom_objects` argument to replace this object.
Exception: code() argument 13 must be str, not int
  warnings.warn(


Modelo cargado OK
Obs space: Box(-1.0, 2.0, (6, 13, 16), float32)


In [4]:
# ── Acceder al env interno para poder hacer render ────────────
# DummyVecEnv -> SB3GymnasiumWrapper -> SMBRamWrapper -> JoypadSpace -> NESEnv
# El render() real está en el NESEnv de nes-py

def get_render_env(vec_env):
    """Extrae el env interno capaz de abrir ventana."""
    # vec_env.envs[0] es SB3GymnasiumWrapper
    e = vec_env.envs[0]
    # Bajar hasta encontrar el env que tiene render con mode='human'
    while hasattr(e, '_env'):
        e = e._env
    while hasattr(e, 'env'):
        e = e.env
    return e

raw_env = get_render_env(vec_env)
print(f'Env para render: {type(raw_env).__name__}')

Env para render: SuperMarioBrosEnv


In [5]:
# ── Jugar con ventana visual ──────────────────────────────────
scores = []

for ep in range(1, EPISODES + 1):
    obs   = vec_env.reset()
    done  = False
    score = 0.0

    while not done:
        # Render: abre/actualiza la ventana del emulador NES
        raw_env.render()

        action, _ = model.predict(obs, deterministic=DETERMINISTIC)
        obs, reward, done, info = vec_env.step(action)
        score += float(reward)
        time.sleep(FRAME_DELAY)

    scores.append(score)
    print(f'Episodio {ep}: score={score:.0f}')

print(f'\nScore medio : {np.mean(scores):.1f}')
print(f'Score máximo: {np.max(scores):.1f}')
print(f'Score mínimo: {np.min(scores):.1f}')

/home/deynar/miniconda3/envs/mario_env/lib/python3.11/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 'numpy.ndarray'>`
  logger.warn(
/home/deynar/miniconda3/envs/mario_env/lib/python3.11/site-packages/gym/utils/passive_env_checker.py:219: DeprecationWarning: WARN: Core environment is written in old step API which returns one bool instead of two. It is recommended to rewrite the environment with new step API. 
  logger.deprecation(
/home/deynar/miniconda3/envs/mario_env/lib/python3.11/site-packages/gym/utils/passive_env_checker.py:225: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(done, (bool, np.bool8)):
/tmp/ipykernel_119802/4242551140.py:15: DeprecationWarning: Conversion of an array with ndim > 0 to a s

Episodio 1: score=239
Episodio 2: score=239
Episodio 3: score=239
Episodio 4: score=239
Episodio 5: score=239

Score medio : 239.0
Score máximo: 239.0
Score mínimo: 239.0


In [6]:
vec_env.close()
print('Listo.')

Listo.
